# Example of using CELIA explainers with a regression model

This notebook demonstrates how to use DiCE (Mothilal et al., 2020), CERTIFAI (Sharma et al., 2019) and NNCE explainer with a regression model using the Celia library. The example uses a pre-trained Bayesian regression model on kidney data.

To run this notebook, you will need to have the trained model and kidney data available. To obtain them, you can go to tests/models/bayesian_train.py and run the script in order to generate the model and the data.

In [13]:
import joblib
import pandas as pd
from celia.model import SklearnModel
from celia.data import PublicData

In [2]:
#Load data
df_train = pd.read_csv('../tests/data/kidney_train.csv')
df_test = pd.read_csv('../tests/data/kidney_test.csv')
X_train, y_train = df_train.drop(columns=['frailty']), df_train['frailty']
X_test, y_test = df_test.drop(columns=['frailty']), df_test['frailty']

#Load model
model = joblib.load('../tests/models/bayesian_model_kidney.pkl')

#targets
targets = model.predict(X_train)
targets = pd.Series(targets, name='target', index=X_train.index)

The next task is to define the "Counterfactual Problem" by specifying which features cannot be changed (immutable) if any, which features are classified as categorical and continous, and what are the feasible values for each feature.

In [3]:
#Metadata
categorical_cols = ['status','sex','disease_type']
immutable_features = ['sex'] # We are only selecting 'sex' as immutable feature.
continuous_features = X_train.columns.difference(categorical_cols).tolist()
feasible_ranges = {'patient': (1,38),
                          'time': (2,562),
                          'status': [0.0, 1.0],
                          'age': (10,69),
                          'sex': [0.0, 1.0],
                          'disease_type': [0.0, 1.0, 2.0, 3.0]}

To ensure a seemesly integration with CELIA, we will wrap our model in a `SklearnModel` object. This is not strictly necessary, but it allows us to use the model in a consistent way with other methods in CELIA.
We will also create a `PublicData` object to hold the data and metadata. This is strictly necessary, as every method in CELIA will obtain the data and constraints from this object.

In [4]:
m = SklearnModel(model)


d = PublicData(data=X_train,
               targets=targets,
               target_name="target",
               column_names=X_train.columns,
               continuous_column_names=continuous_features,
               categorical_column_names=categorical_cols,
               immutable_column_names=immutable_features,
               feasible_values=feasible_ranges,)


Now, we will create our explainer. We will use the `DiceRegressorExplainer` class, which is specifically designed for regression models. We will use the 'random' method to generate counterfactuals.

In [5]:
from celia.explainers import DiceRegressorExplainer, CertifaiRegressorExplainer, NNCERegressorExplainer
dice = DiceRegressorExplainer(data=d, model=m, method='random')
certifai = CertifaiRegressorExplainer(data=d, model=m)
nnce = NNCERegressorExplainer(data=d, model=m)

Now we will select an instance to explain. For demonstration purposes, we will use instance = 1 from the test set.

In [6]:
itbe = X_test.iloc[1].to_frame().T
print(f'Predicted value by the model: {model.predict(itbe)} | Ground truth: {y_test.iloc[1]}')
itbe

Predicted value by the model: [1.14246993] | Ground truth: 0.8


,patient,time,status,age,sex,disease_type
1,18,114,1,44,0,2


### DiCE Counterfactuals

In [7]:
dice_result = dice.generate_counterfactuals(itbe, target_range=[0.5,1], total_CFs=1)
dice_result

,patient,time,status,age,sex,disease_type,target
0,18,388,1,44,0,2,1.0


### CERTIFAI Counterfactuals

#### Note
Depending on the method is what attributes will be respected during the generation of counterfactuals. For example, `CertifaiRegressorExplainer` method does not include feasible ranges in its design, so it will not respect them.

In [8]:
certifai_result = certifai.generate_counterfactuals(itbe, target_range=[0,1], trained_with_columns=True)
certifai_result

/Users/alan.paredes/Projects/celia/.venv/lib/python3.12/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but BayesianRidge was fitted with feature names
  warnings.warn(
/Users/alan.paredes/Projects/celia/.venv/lib/python3.12/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but BayesianRidge was fitted with feature names
  warnings.warn(
/Users/alan.paredes/Projects/celia/.venv/lib/python3.12/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but BayesianRidge was fitted with feature names
  warnings.warn(
/Users/alan.paredes/Projects/celia/.venv/lib/python3.12/site-packages/sklearn/base.py:465: UserWarning: X does not have valid feature names, but BayesianRidge was fitted with feature names
  warnings.warn(


,patient,time,status,age,sex,disease_type,target
0,29.0,194.0,1.0,48.0,0.0,0.0,0.984082


### NNCE Counterfactuals

NNCE stands for Nearest Neighbors Counterfactual Explanations. This method uses the nearest neighbors of the instance to generate counterfactuals. Its main limitation is that if we set complex constraints to the Counterfactual Problem, it may not be able to find counterfactuals that respect them.

In [9]:
nnce_result = nnce.generate_counterfactuals(itbe, target_range=[0,1])

As we can see, NNCE could not find any counterfactual from the training set that has a target between 0 and 1 where `sex` remains the same.
But what if we remove the `sex` feature from the immutable features? In this case, NNCE will be able to find counterfactuals that respect the feasible ranges and the target range.

In [10]:
d = PublicData(data=X_train,
               targets=targets,
               target_name="target",
               column_names=X_train.columns,
               continuous_column_names=continuous_features,
               categorical_column_names=categorical_cols,
               immutable_column_names=None,
               feasible_values=feasible_ranges,)
nnce_no_immutables = NNCERegressorExplainer(data=d, model=m)
nnce_no_immutables_results = nnce_no_immutables.generate_counterfactuals(itbe, target_range=[0,1])
nnce_no_immutables_results

,patient,time,status,age,sex,disease_type,target
0,17,185,1,60,0,2,0.98882
